# YOLO Assignment – Subtask 2: Object Counting in Video

**Task:** Detect, track, and count objects crossing a defined line in a video.

## Objective
This notebook builds a simple video-counting system using a pretrained YOLO model. It detects objects, tracks their movement, counts objects crossing a line, and saves an annotated output video.

## Application (Real-World Use)
Object counting is useful in:
- **Traffic monitoring:** count cars, buses, trucks, and motorcycles passing a road point.
- **People counting:** estimate how many people enter or leave a place.
- **Retail analytics:** count customers passing a selected area.
- **Smart cities:** collect traffic-flow information.

## Important: Do I need a Roboflow dataset?
No. This subtask asks us to use a **pretrained YOLO model** and a video. It does not require downloading a dataset or training a new model. We use Roboflow's **PolygonZone** website only to obtain the coordinates of the counting line.

### Where to get each required item
| Item | Where to get it |
|---|---|
| Pretrained YOLO model | Ultralytics automatically downloads the model weights when we load `yolo26n.pt` |
| Input video | Instructor-provided video, your own recording, or an openly licensed video file you are permitted to use |
| Counting-line coordinates | Roboflow PolygonZone: https://polygonzone.roboflow.com/ |
| Object-counting instructions | https://docs.ultralytics.com/guides/object-counting/ |

**Video tip:** choose a fixed-camera traffic or people video where objects clearly cross one line. Upload an actual video file such as MP4. A YouTube watch-page URL is not itself a video file; use a file supplied by the creator or another authorized source.

## 1. Prepare the Environment


- **Ultralytics:** provides the pretrained YOLO model and ObjectCounter solution.
- **OpenCV:** opens the video, reads frames, obtains video properties, and writes the output video.
- **Matplotlib:** displays a frame so we can choose a suitable counting-line location.

In [2]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [13]:
pip install opencv-python

### import library

In [12]:
from ultralytics import YOLO
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## 2. Load a Pretrained YOLO Model

A **pretrained model** has already learned to recognize common object classes from a large labelled dataset. We use the lightweight YOLO26 Nano model (`yolo26n.pt`) as an example. Ultralytics downloads its weights automatically the first time the model is loaded.

We do not train a new model in this task. The model detects common COCO classes, including person, bicycle, car, motorcycle, bus, and truck.

In [10]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("/content/Traffic.mp4")
assert cap.isOpened(), "Error reading video file"

# region_points = [(20, 400), (1080, 400)]                                      # line counting
region_points = [(20, 400), (1080, 400), (1080, 360), (20, 360)]  # rectangular region
# region_points = [(20, 400), (1080, 400), (1080, 360), (20, 360), (20, 400)]   # polygon region

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("object_counting_output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Initialize object counter object
counter = solutions.ObjectCounter(
    show=True,  # display the output
    region=region_points,  # pass region points
    model="yolo26s.pt",  # model="yolo26n-obb.pt" for object counting with OBB model.
    # classes=[0, 2],  # count specific classes, e.g., person and car with the COCO pretrained model.
    # tracker="botsort.yaml",  # choose trackers, e.g., "bytetrack.yaml"
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = counter(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)  # write the processed frame.

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows

Ultralytics Solutions: ✅ {'source': None, 'model': 'yolo26s.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': [(20, 400), (1080, 400), (1080, 360), (20, 360)], 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90.0, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': True, 'iou': None, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'tracktrack.yaml', 'verbose': True, 'data': 'images', 'recognizer': 'paddleocr'}
0: 640x640 2.8ms, 5 car
Speed: 161.4ms track, 2.8ms solution per image at shape (1, 3, 640, 640)

1: 640x640 1.2ms, 4 car
Speed: 32.0ms track, 1.2ms solution per image at shape (1, 3, 640, 640)

2: 640x640 1.4ms, 4 car
Speed: 29.9ms track, 1

### Upload a Traffic or Object-Movement Video

### Read the Video with OpenCV

OpenCV's `VideoCapture` opens the video so that we can read it frame by frame.

We extract:
- **Width:** horizontal pixel count in each frame.
- **Height:** vertical pixel count in each frame.
- **FPS:** frames per second.
- **Frame count:** total number of frames in the video.

We must check `isOpened()` because a wrong path, unsupported format, or damaged video can prevent OpenCV from opening it.


## 4. Define the Counting Line Using Roboflow PolygonZone

A **counting line** is an imaginary line placed across the video frame. The counter detects when a tracked object crosses this line.

The line is represented by two points:
- Point 1: `(x1, y1)`
- Point 2: `(x2, y2)`